In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dataset/processed/final_dataset_preprocessed.csv')

# Add explicit row-order index — preserves temporal ordering guarantee
df['time_index'] = range(len(df))

# Move to front for visibility
cols = ['time_index'] + [c for c in df.columns if c != 'time_index']
df = df[cols]

print(df[['time_index', 'month', 'day_of_week']].head(5))
print(f"\nShape: {df.shape}")

   time_index  month  day_of_week
0           0      1            2
1           1      1            3
2           2      1            4
3           3      1            5
4           4      1            6

Shape: (9132, 31)


In [ ]:
# Verify lag features don't carry future information
# temp_lag3 should equal temperature from 3 rows earlier

lag_check = pd.DataFrame({
    'temp_now'    : df['temperature'].values,
    'temp_lag3'   : df['temp_lag3'].values,
    'temp_3_ago'  : df['temperature'].shift(3).values,
})

# If lag was computed correctly, temp_lag3 == temperature.shift(3)
match = (lag_check['temp_lag3'] - lag_check['temp_3_ago']).abs()
print("Max difference between temp_lag3 and temperature.shift(3):", match.max())
print("Mean difference:", match.mean())

if match.max() < 1e-6:
    print("\nLag features are correctly computed — no leakage detected")
else:
    print("\n  Lag features do NOT match shift(3) — possible leakage, recompute below")

Max difference between temp_lag3 and temperature.shift(3): 18.78
Mean difference: 0.010897140979296746

⚠️  Lag features do NOT match shift(3) — possible leakage, recompute below


In [3]:
# Recompute lag features from scratch to be safe
# Ensures correct temporal ordering, no leakage

df = df.sort_values('time_index').reset_index(drop=True)

for lag in [1, 2, 3]:
    df[f'temp_lag{lag}']     = df['temperature'].shift(lag)
    df[f'humidity_lag{lag}'] = df['humidity'].shift(lag)

# Rolling features (3-day window, only uses past — no future leak)
df['temp_rolling3_mean'] = df['temperature'].rolling(3, min_periods=1).mean()
df['temp_rolling3_std']  = df['temperature'].rolling(3, min_periods=1).std().fillna(0)
df['hum_rolling3_mean']  = df['humidity'].rolling(3, min_periods=1).mean()

# Change features (day-over-day delta)
df['temp_delta']     = df['temperature'].diff().fillna(0)
df['humidity_delta'] = df['humidity'].diff().fillna(0)

# Drop first 3 rows where lag is NaN (only 3 rows out of 9132 — negligible loss)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape after recomputing lags: {df.shape}")
print(f"Rows dropped (lag NaN): {9132 - len(df)}")
print(df[['temperature','temp_lag1','temp_lag2','temp_lag3',
          'temp_rolling3_mean','temp_delta']].head(6))

Shape after recomputing lags: (9129, 40)
Rows dropped (lag NaN): 3
   temperature  temp_lag1  temp_lag2  temp_lag3  temp_rolling3_mean  \
0        24.46      24.06      23.29      23.05           23.936667   
1        24.23      24.46      24.06      23.29           24.250000   
2        23.11      24.23      24.46      24.06           23.933333   
3        22.39      23.11      24.23      24.46           23.243333   
4        22.20      22.39      23.11      24.23           22.566667   
5        22.59      22.20      22.39      23.11           22.393333   

   temp_delta  
0        0.40  
1       -0.23  
2       -1.12  
3       -0.72  
4       -0.19  
5        0.39  


In [4]:
# drought_index: the one domain feature that ISN'T in your data yet
# High temp + low water level = drought stress

df['drought_index'] = df['temperature'] / (df['water_level'] + 1e-5)

print("drought_index stats:")
print(df['drought_index'].describe().round(4))

# Quick sanity: high drought when water_level is low and temp is high
check = df[['temperature', 'water_level', 'drought_index']].sort_values('drought_index', ascending=False)
print("\nTop 5 drought rows:")
print(check.head())

drought_index stats:
count    9.129000e+03
mean     1.700868e+04
std      2.173426e+05
min      7.860000e-02
25%      2.424000e-01
50%      2.914000e-01
75%      3.770100e+00
max      3.126000e+06
Name: drought_index, dtype: float64

Top 5 drought rows:
      temperature  water_level  drought_index
6111        31.26          0.0      3126000.0
6112        31.22          0.0      3122000.0
8395        30.58          0.0      3058000.0
3829        30.43          0.0      3043000.0
3828        30.25          0.0      3025000.0


In [5]:
from collections import Counter

counts = Counter(df['target'])
total  = len(df)

print("Class distribution:")
for k, v in sorted(counts.items()):
    print(f"  class {k}: {v:,}  ({v/total*100:.1f}%)")

ratio = max(counts.values()) / min(counts.values())
print(f"\nImbalance ratio: {ratio:.2f}x")

# Compute class weights for use in model training
# sklearn / XGBoost / PyTorch all accept these
class_weights = {cls: total / (len(counts) * cnt) for cls, cnt in counts.items()}
print(f"\nClass weights: {class_weights}")

# For sklearn: class_weight=class_weights
# For XGBoost: scale_pos_weight = counts[0] / counts[1]
scale_pos_weight = counts[0] / counts[1]
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")

if ratio < 3:
    print("\n✅ Ratio < 3x — class weights are sufficient, SMOTE not necessary")
else:
    print("\n⚠️  Ratio > 3x — consider SMOTE or oversampling")

Class distribution:
  class 0: 3,779  (41.4%)
  class 1: 5,350  (58.6%)

Imbalance ratio: 1.42x

Class weights: {0: 1.2078592220164064, 1: 0.853177570093458}
XGBoost scale_pos_weight: 0.7064

✅ Ratio < 3x — class weights are sufficient, SMOTE not necessary


In [6]:
# CRITICAL: never shuffle time-series data before splitting
# Use chronological order: train → val → test

n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols].values
y = df['target'].values

X_train, y_train = X[:train_end],  y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],    y[val_end:]

print(f"Train : {X_train.shape}  |  target balance: {y_train.mean():.2%}")
print(f"Val   : {X_val.shape}    |  target balance: {y_val.mean():.2%}")
print(f"Test  : {X_test.shape}   |  target balance: {y_test.mean():.2%}")

Train : (6390, 40)  |  target balance: 54.62%
Val   : (1369, 40)    |  target balance: 55.59%
Test  : (1370, 40)   |  target balance: 80.22%


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on train
X_val_scaled   = scaler.transform(X_val)          # transform val
X_test_scaled  = scaler.transform(X_test)         # transform test

print("Scaling done (fit on train only — no leakage)")
print(f"X_train mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"X_train std  (should be ~1): {X_train_scaled.std():.4f}")
print(f"X_val   mean (will differ) : {X_val_scaled.mean():.4f}")

# Save scaler for inference
import joblib
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved → scaler.pkl")

Scaling done (fit on train only — no leakage)
X_train mean (should be ~0): 0.0000
X_train std  (should be ~1): 1.0000
X_val   mean (will differ) : 0.0233
Scaler saved → scaler.pkl


In [8]:
def make_sequences(X, y, time_steps=7):
    """
    Convert tabular data to LSTM input: [samples, time_steps, features]
    Each sample is a sliding window of `time_steps` consecutive rows.
    Label = target at the LAST row of the window.
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i : i + time_steps])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 7   # 7-day look-back window

X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train, TIME_STEPS)
X_val_seq,   y_val_seq   = make_sequences(X_val_scaled,   y_val,   TIME_STEPS)
X_test_seq,  y_test_seq  = make_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f"LSTM input shapes:")
print(f"  X_train : {X_train_seq.shape}   → [samples, time_steps, features]")
print(f"  X_val   : {X_val_seq.shape}")
print(f"  X_test  : {X_test_seq.shape}")
print(f"\n  Verify: axis 1 = {X_train_seq.shape[1]} (time steps)")
print(f"          axis 2 = {X_train_seq.shape[2]} (features)")

LSTM input shapes:
  X_train : (6383, 7, 40)   → [samples, time_steps, features]
  X_val   : (1362, 7, 40)
  X_test  : (1363, 7, 40)

  Verify: axis 1 = 7 (time steps)
          axis 2 = 40 (features)


In [9]:
import joblib

# Save preprocessed flat CSV (for XGBoost / Random Forest)
df.to_csv('final_dataset_preprocessed_v2.csv', index=False)

# Save numpy arrays (for LSTM / PyTorch / TF)
np.save('X_train_seq.npy', X_train_seq)
np.save('X_val_seq.npy',   X_val_seq)
np.save('X_test_seq.npy',  X_test_seq)
np.save('y_train_seq.npy', y_train_seq)
np.save('y_val_seq.npy',   y_val_seq)
np.save('y_test_seq.npy',  y_test_seq)

print("Saved:")
print("  ✅ final_dataset_preprocessed_v2.csv  (flat, for tree models)")
print("  ✅ X_train_seq.npy / y_train_seq.npy  (LSTM train)")
print("  ✅ X_val_seq.npy   / y_val_seq.npy    (LSTM val)")
print("  ✅ X_test_seq.npy  / y_test_seq.npy   (LSTM test)")
print("  ✅ scaler.pkl                          (StandardScaler)")
print(f"\nFinal feature count : {X_train_seq.shape[2]}")
print(f"Time steps (window) : {X_train_seq.shape[1]}")
print(f"Training samples    : {X_train_seq.shape[0]:,}")

Saved:
  ✅ final_dataset_preprocessed_v2.csv  (flat, for tree models)
  ✅ X_train_seq.npy / y_train_seq.npy  (LSTM train)
  ✅ X_val_seq.npy   / y_val_seq.npy    (LSTM val)
  ✅ X_test_seq.npy  / y_test_seq.npy   (LSTM test)
  ✅ scaler.pkl                          (StandardScaler)

Final feature count : 40
Time steps (window) : 7
Training samples    : 6,383
